In [ ]:
# Cell 1: Install tools and NLTK for English Detection
!pip install -q indic-nlp-library nltk

import nltk
print("Downloading English Dictionary...")
nltk.download('words', quiet=True)
print("Setup complete!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 4.4 MB/s eta 0:00:00
Setup complete!


In [ ]:
# Cell 2: Hindi Number Normalization Logic
import re

# Comprehensive list of spoken Hindi numbers 1-99
hindi_numbers = {
    'शून्य': 0, 'जीरो': 0, 'एक': 1, 'दो': 2, 'तीन': 3, 'चार': 4, 'पांच': 5, 'पाँच': 5,
    'छह': 6, 'सात': 7, 'आठ': 8, 'नौ': 9, 'दस': 10, 'ग्यारह': 11, 'बारह': 12, 'तेरह': 13,
    'चौदह': 14, 'पंद्रह': 15, 'सोलह': 16, 'सत्रह': 17, 'अठारह': 18, 'उन्नाइस': 19, 'उन्नीस': 19,
    'बीस': 20, 'इक्कीस': 21, 'बाईस': 22, 'तेईस': 23, 'चौबीस': 24, 'पच्चीस': 25, 'छब्बीस': 26,
    'सत्ताइस': 27, 'अट्ठाइस': 28, 'उनतीस': 29, 'तीस': 30, 'इकतीस': 31, 'बत्तीस': 32, 'तैंतीस': 33,
    'चौंतीस': 34, 'पैंतीस': 35, 'छत्तीस': 36, 'सैंतीस': 37, 'अड़तीस': 38, 'उनतालीस': 39, 'चालीस': 40,
    'इकतालीस': 41, 'बयालीस': 42, 'तैंतालीस': 43, 'चवालीस': 44, 'पैंतालीस': 45, 'छियालीस': 46,
    'सैंतालीस': 47, 'अड़तालीस': 48, 'उनचास': 49, 'पचास': 50, 'इक्यावन': 51, 'बावन': 52, 'तिरेपन': 53,
    'चौवन': 54, 'पचपन': 55, 'छप्पन': 56, 'सत्तावन': 57, 'अट्ठावन': 58, 'उनसठ': 59, 'साठ': 60,
    'इक्सठ': 61, 'बासठ': 62, 'तिरेसठ': 63, 'चौंसठ': 64, 'पैंसठ': 65, 'छियासठ': 66, 'सड़सठ': 67,
    'अड़सठ': 68, 'उनहत्तर': 69, 'सत्तर': 70, 'इकहत्तर': 71, 'बहत्तर': 72, 'तिहत्तर': 73, 'चौहत्तर': 74,
    'पचहत्तर': 75, 'छिहत्तर': 76, 'सतहत्तर': 77, 'अठहत्तर': 78, 'उनासी': 79, 'अस्सी': 80, 'इक्यासी': 81,
    'बयासी': 82, 'तिरासी': 83, 'चौरासी': 84, 'पचासी': 85, 'छियासी': 86, 'सत्तासी': 87, 'अट्ठासी': 88,
    'नवासी': 89, 'नब्बे': 90, 'इक्यानवे': 91, 'बानवे': 92, 'तिरानवे': 93, 'चौरानवे': 94, 'पचानवे': 95,
    'छियानवे': 96, 'सत्तानवे': 97, 'अट्ठानवे': 98, 'निन्यानवे': 99
}

hindi_multipliers = {
    'सौ': 100, 'साै': 100,
    'हज़ार': 1000, 'हजार': 1000,
    'लाख': 100000,
    'करोड़': 10000000, 'करोड': 10000000
}

# Cell 2: Hindi Number Normalization Logic
# (Keep hindi_numbers and hindi_multipliers dictionaries exactly as they were before)

def normalize_hindi_numbers(text):
    words = text.split()
    output = []
    current_number_block = []

    def process_number_block(block):
        if not block: return ""
        total = 0
        temp = 0
        for w in block:
            if w in hindi_numbers:
                temp += hindi_numbers[w]
            elif w in hindi_multipliers:
                if temp == 0: temp = 1
                total += temp * hindi_multipliers[w]
                temp = 0
        total += temp
        return str(total)

    for word in words:
        # FIX: Directly strip only standard punctuation, protecting all Hindi Matras!
        clean_word = word.strip('.,?!।()"\'')

        if clean_word in hindi_numbers or clean_word in hindi_multipliers:
            current_number_block.append(clean_word)
        else:
            if current_number_block:
                final_digit = process_number_block(current_number_block)
                output.append(final_digit)
                current_number_block = []
            output.append(word)

    if current_number_block:
        output.append(process_number_block(current_number_block))

    return " ".join(output)

print("Number Normalization Pipeline Loaded (Fixed)!")



Number Normalization Pipeline Loaded (Fixed)!


In [ ]:
# Cell 3: English Transliteration Detection
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator
from nltk.corpus import words

en_words = set(words.words())

def tag_english_words(text):
    words_list = text.split()
    tagged_sentence = []

    for word in words_list:
        # FIX: Safely strip standard punctuation
        clean_word = word.strip('.,?!।()"\'')

        if clean_word.isdigit() or not clean_word:
            tagged_sentence.append(word)
            continue

        try:
            transliterated = UnicodeIndicTransliterator.transliterate(clean_word, "hi", "itrans").lower()

            if transliterated in en_words:
                tagged_word = word.replace(clean_word, f"[EN]{clean_word}[/EN]")
                tagged_sentence.append(tagged_word)
            else:
                tagged_sentence.append(word)
        except:
            tagged_sentence.append(word)

    return " ".join(tagged_sentence)

print("English Tagging Pipeline Loaded (Fixed)!")


English Tagging Pipeline Loaded (Fixed)!


In [ ]:
# Cell 3: English Word Tagging (Using a Targeted Lexicon)
import re

# In a pure-python pipeline, we maintain a dictionary of English
# words commonly spoken in Devanagari (Hinglish Loanwords).
english_loanwords_in_hindi = {
    'इंटरव्यू', 'जॉब', 'मैम', 'सर', 'प्रॉब्लम', 'सॉल्व', 'कंप्यूटर',
    'कॉल', 'फोन', 'टाइम', 'वीडियो', 'ऑनलाइन', 'ऑफिस', 'स्कूल',
    'कॉलेज', 'क्लास', 'टेस्ट', 'पास', 'फेल', 'नंबर', 'मैसेज', 'लाइफ',
    'फ्रेंड', 'सोशल', 'मीडिया', 'मार्केट', 'शेयर', 'कार', 'बाइक', 'रोड'
}

def tag_english_words(text):
    words_list = text.split()
    tagged_sentence = []

    for word in words_list:
        # Strip standard punctuation safely
        clean_word = word.strip('.,?!।()"\'')

        # If it's a number or space, skip it
        if clean_word.isdigit() or not clean_word:
            tagged_sentence.append(word)
            continue

        # If the word is in our English Loanword Lexicon, tag it!
        if clean_word in english_loanwords_in_hindi:
            # Wrap the original word with [EN] [/EN]
            tagged_word = word.replace(clean_word, f"[EN]{clean_word}[/EN]")
            tagged_sentence.append(tagged_word)
        else:
            tagged_sentence.append(word)

    return " ".join(tagged_sentence)

print("English Tagging Pipeline Loaded (Lexicon Based)!")


English Tagging Pipeline Loaded (Lexicon Based)!


In [ ]:
# Cell 4: Test the Pipeline
test_sentences = [
    "मेरे पास दो हैं",                                       # Simple: 2
    "मेरे पास दस और सौ हैं",                                 # Simple: 10, 100
    "मेरे पास तीन सौ चौवन रुपये हैं",                         # Compound: 354
    "यह पच्चीस और एक हज़ार है",                              # Compound: 25, 1000
    "हमेशा दो-चार बातें ही होती हैं",                        # Edge Case Idiom: "दो-चार" should stay identical!
    "मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई"        # English Detection Example
]

print("=== RUNNING THE TEXT CLEANUP PIPELINE ===")
for sentence in test_sentences:
    print(f"\nOriginal Phrase: {sentence}")

    # 1. Run Number Normalization
    normalized = normalize_hindi_numbers(sentence)

    # 2. Run English Detection on the result
    final_output = tag_english_words(normalized)

    print(f"Final Cleaned  : {final_output}")


=== RUNNING THE TEXT CLEANUP PIPELINE ===

Original Phrase: मेरे पास दो हैं
Final Cleaned  : मेरे [EN]पास[/EN] 2 हैं

Original Phrase: मेरे पास दस और सौ हैं
Final Cleaned  : मेरे [EN]पास[/EN] 10 और 100 हैं

Original Phrase: मेरे पास तीन सौ चौवन रुपये हैं
Final Cleaned  : मेरे [EN]पास[/EN] 354 रुपये हैं

Original Phrase: यह पच्चीस और एक हज़ार है
Final Cleaned  : यह 25 और 1000 है

Original Phrase: हमेशा दो-चार बातें ही होती हैं
Final Cleaned  : हमेशा दो-चार बातें ही होती हैं

Original Phrase: मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई
Final Cleaned  : मेरा [EN]इंटरव्यू[/EN] बहुत अच्छा गया और मुझे [EN]जॉब[/EN] मिल गई


In [ ]:
!pip install -q transformers datasets librosa torch indic-nlp-library nltk

import warnings
warnings.filterwarnings("ignore") # This hides ugly HuggingFace warnings!

import pandas as pd
import urllib.request
import json
import torch
import librosa
import re
from transformers import pipeline
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator

# ==============================================================
# 0. THE TEXT CLEANUP PIPELINE LOGIC
# ==============================================================

hindi_numbers = {
    'शून्य': 0, 'जीरो': 0, 'एक': 1, 'दो': 2, 'तीन': 3, 'चार': 4, 'पांच': 5, 'पाँच': 5,
    'छह': 6, 'सात': 7, 'आठ': 8, 'नौ': 9, 'दस': 10, 'ग्यारह': 11, 'बारह': 12, 'तेरह': 13,
    'चौदह': 14, 'पंद्रह': 15, 'सोलह': 16, 'सत्रह': 17, 'अठारह': 18, 'उन्नाइस': 19, 'उन्नीस': 19,
    'बीस': 20, 'इक्कीस': 21, 'बाईस': 22, 'तेईस': 23, 'चौबीस': 24, 'पच्चीस': 25, 'छब्बीस': 26,
    'सत्ताइस': 27, 'अट्ठाइस': 28, 'उनतीस': 29, 'तीस': 30, 'इकतीस': 31, 'बत्तीस': 32, 'तैंतीस': 33,
    'चौंतीस': 34, 'पैंतीस': 35, 'छत्तीस': 36, 'सैंतीस': 37, 'अड़तीस': 38, 'उनतालीस': 39, 'चालीस': 40,
    'इकतालीस': 41, 'बयालीस': 42, 'तैंतालीस': 43, 'चवालीस': 44, 'पैंतालीस': 45, 'छियालीस': 46,
    'सैंतालीस': 47, 'अड़तालीस': 48, 'उनचास': 49, 'पचास': 50, 'इक्यावन': 51, 'बावन': 52, 'तिरेपन': 53,
    'चौवन': 54, 'पचपन': 55, 'छप्पन': 56, 'सत्तावन': 57, 'अट्ठावन': 58, 'उनसठ': 59, 'साठ': 60,
    'इक्सठ': 61, 'बासठ': 62, 'तिरेसठ': 63, 'चौंसठ': 64, 'पैंसठ': 65, 'छियासठ': 66, 'सड़सठ': 67,
    'अड़सठ': 68, 'उनहत्तर': 69, 'सत्तर': 70, 'इकहत्तर': 71, 'बहत्तर': 72, 'तिहत्तर': 73, 'चौहत्तर': 74,
    'पचहत्तर': 75, 'छिहत्तर': 76, 'सतहत्तर': 77, 'अठहत्तर': 78, 'उनासी': 79, 'अस्सी': 80, 'इक्यासी': 81,
    'बयासी': 82, 'तिरासी': 83, 'चौरासी': 84, 'पचासी': 85, 'छियासी': 86, 'सत्तासी': 87, 'अट्ठासी': 88,
    'नवासी': 89, 'नब्बे': 90, 'इक्यानवे': 91, 'बानवे': 92, 'तिरानवे': 93, 'चौरानवे': 94, 'पचानवे': 95,
    'छियानवे': 96, 'सत्तानवे': 97, 'अट्ठानवे': 98, 'निन्यानवे': 99
}

hindi_multipliers = {'सौ': 100, 'साै': 100, 'हज़ार': 1000, 'हजार': 1000, 'लाख': 100000, 'करोड़': 10000000, 'करोड': 10000000}

english_loanwords_in_hindi = {
    'इंटरव्यू', 'जॉब', 'मैम', 'सर', 'प्रॉब्लम', 'सॉल्व', 'कंप्यूटर',
    'कॉल', 'फोन', 'टाइम', 'वीडियो', 'ऑनलाइन', 'ऑफिस', 'स्कूल',
    'कॉलेज', 'क्लास', 'टेस्ट', 'पास', 'फेल', 'नंबर', 'मैसेज', 'लाइफ',
    'फ्रेंड', 'सोशल', 'मीडिया', 'मार्केट', 'शेयर', 'कार', 'बाइक', 'रोड'
}

def normalize_hindi_numbers(text):
    words = text.split()
    output = []
    current_number_block = []
    def process_number_block(block):
        if not block: return ""
        total, temp = 0, 0
        for w in block:
            if w in hindi_numbers: temp += hindi_numbers[w]
            elif w in hindi_multipliers:
                if temp == 0: temp = 1
                total += temp * hindi_multipliers[w]
                temp = 0
        total += temp
        return str(total)

    for word in words:
        clean_word = word.strip('.,?!।()"\'')
        if clean_word in hindi_numbers or clean_word in hindi_multipliers:
            current_number_block.append(clean_word)
        else:
            if current_number_block:
                output.append(process_number_block(current_number_block))
                current_number_block = []
            output.append(word)
    if current_number_block: output.append(process_number_block(current_number_block))
    return " ".join(output)

def tag_english_words(text):
    words_list = text.split()
    tagged_sentence = []
    for word in words_list:
        clean_word = word.strip('.,?!।()"\'')
        if clean_word in english_loanwords_in_hindi:
            tagged_sentence.append(word.replace(clean_word, f"[EN]{clean_word}[/EN]"))
        else:
            tagged_sentence.append(word)
    return " ".join(tagged_sentence)


# ==============================================================
# 1. SETUP THE INFERENCE LOOP
# ==============================================================

print("1. Downloading Ground Truth Dataset...")
sheet_url = "https://docs.google.com/spreadsheets/d/1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI/export?format=csv"
df = pd.read_csv(sheet_url)

print("2. Booting up Whisper-Small (Heavy AI Model) on GPU...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
    chunk_length_s=30,  # Limits memory constraints
    return_timestamps=False
)

# Limit to 3 files so your Colab doesn't time out
test_limit = 3

print(f"\n3. Beginning Audio Transcription and Cleanup (Testing first {test_limit} files)...\n")
for index, row in df.head(test_limit).iterrows():

    # APPLY FIX TO ASSIGNMENT PDF URLS
    raw_audio_url = row['rec_url_gcp'].replace("joshtalks-data-collection/hq_data/hi/", "upload_goai/")
    raw_json_url = row['transcription_url_gcp'].replace("joshtalks-data-collection/hq_data/hi/", "upload_goai/")

    try:
        # A) Human Ground Truth
        json_response = urllib.request.urlopen(raw_json_url)
        ground_truth_data = json.loads(json_response.read().decode('utf-8'))

        # Combine JSON chunks correctly
        if isinstance(ground_truth_data, list):
            human_text = " ".join([segment.get('text', '') for segment in ground_truth_data])
        else:
            human_text = ground_truth_data.get('text', "Unknown")

        # B) Download Audio
        audio_filename = f"temp_audio_{index}.mp3"
        urllib.request.urlretrieve(raw_audio_url, audio_filename)

        # C) Transcribe with Whisper!
        audio_array, sampling_rate = librosa.load(audio_filename, sr=16000)
        whisper_output = transcriber({"sampling_rate": sampling_rate, "raw": audio_array})
        messy_text = whisper_output["text"]

        # D) Feed through Cleanup Pipeline!
        normalized_numbers_text = normalize_hindi_numbers(messy_text)
        final_clean_text = tag_english_words(normalized_numbers_text)

        print("\n" + "="*80)
        print(f"✅ Audio File {index+1} / 3 processed!")
        print(f"Human Reference Text   : {human_text}")
        print(f"Whisper Raw Messy Text : {messy_text}")
        print(f"PIPELINE CLEANED TEXT  : {final_clean_text}")
        print("="*80)

    except Exception as e:
        print(f"Failed on file {index+1}: {e}")

print("\nQuestion 2 Pipeline Complete!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 14.6 MB/s eta 0:00:00
1. Downloading Ground Truth Dataset...
2. Booting up Whisper-Small (Heavy AI Model) on GPU...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



3. Beginning Audio Transcription and Cleanup (Testing first 3 files)...


✅ Audio File 1 / 3 processed!
Human Reference Text   : अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे पहली बारी था क्योंकि चलना नहीं आता न वहाँ का जो लैंड एरिया होता है इधर उधर काफी इधर जाओ तो उधर लुढ़क जाओगे हां तो फिर वहां जो दिन भर तो दिन में तो खोजने में वक्त बीत गया। जब रात की बारी आई तो हमने टेंट गड़ा और रहा तो जब पता जैसी रात हुआ ना शाम मतलब छै सात में इतना अजीब सा आवाज आने लगा बहुत अजीब सा डर तो इतना लगा था कि अगर कोई आता तो हम

In [ ]:
!pip install -q transformers datasets librosa torch

import pandas as pd
import urllib.request
import json
import torch
import librosa
from transformers import pipeline

# --- STEP 1: LOAD THE DATASET ---
print("1. Downloading Ground Truth Dataset...")
# This is the URL to the 10-hour dataset sheet provided in the assignment PDF
sheet_url = "https://docs.google.com/spreadsheets/d/1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI/export?format=csv"
df = pd.read_csv(sheet_url)

# --- STEP 2: LOAD WHISPER-SMALL TO GPU ---
print("2. Booting up Whisper-Small (Heavy AI Model) on GPU...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"

transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
    chunk_length_s=30  # <--- THIS MAGICAL LINE FIXES THE CRASH!
)


# --- STEP 3: RUN THE INFERENCE LOOP ON THE FIRST 3 AUDI FILES ---
test_limit = 3

print(f"\n3. Beginning Audio Transcription and Cleanup (Testing first {test_limit} files)...\n")
for index, row in df.head(test_limit).iterrows():

    # ⚠️ APPLYING THE ASSIGNMENT PDF URL FIX
    raw_audio_url = row['rec_url_gcp'].replace("joshtalks-data-collection/hq_data/hi/", "upload_goai/")
    raw_json_url = row['transcription_url_gcp'].replace("joshtalks-data-collection/hq_data/hi/", "upload_goai/")

    try:
        # A) Download and parse what the Human actually wrote (Ground Truth)
        json_response = urllib.request.urlopen(raw_json_url)
        ground_truth_data = json.loads(json_response.read().decode('utf-8'))

        # FIX: Combine the text from all the audio segments in the list!
        if isinstance(ground_truth_data, list):
            human_text = " ".join([segment.get('text', '') for segment in ground_truth_data])
        else:
            human_text = ground_truth_data.get('text', "Unknown")

        # B) Download the actual .wav / .mp3 audio file locally
        audio_filename = f"temp_audio_{index}.mp3"
        urllib.request.urlretrieve(raw_audio_url, audio_filename)

        # C) Run the audio through the Heavy Whisper AI to get the "Raw Messy Output"
        audio_array, sampling_rate = librosa.load(audio_filename, sr=16000)
        whisper_output = transcriber({"sampling_rate": sampling_rate, "raw": audio_array})
        messy_text = whisper_output["text"]

        # D) Feed Whisper's messy text into our Text-Cleanup Pipeline!
        normalized_numbers_text = normalize_hindi_numbers(messy_text)
        final_clean_text = tag_english_words(normalized_numbers_text)

        print("="*60)
        print(f"✅ Audio File {index+1} ({row['recording_id']}) processed!")
        print(f"Human Reference Text : {human_text}")
        print(f"Whisper Messy Text   : {messy_text}")
        print(f"PIPELINE CLEANED TEXT: {final_clean_text}")
        print("="*60)

    except Exception as e:
        print(f"Failed on file {index+1}: {e}")

print("Question 2 Pipeline Complete!")



1. Downloading Ground Truth Dataset...
2. Booting up Whisper-Small (Heavy AI Model) on GPU...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



3. Beginning Audio Transcription and Cleanup (Testing first 3 files)...



Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

Failed on file 1: name 'normalize_hindi_numbers' is not defined
Failed on file 2: name 'normalize_hindi_numbers' is not defined
Failed on file 3: name 'normalize_hindi_numbers' is not defined
Question 2 Pipeline Complete!
